# The `abaqus` driver — reading a finite-element part

**What the driver does.** An Abaqus **`.inp`** file is the input deck for a structural finite-element-analysis
(FEA) model: it lists the model's **nodes** (points) and **elements** (cells). The `abaqus` driver parses it and
builds a pyramids `MeshDataset` — the common object every pyramids-mesh driver returns — which you then explore
and plot with the ordinary pyramids API.

**The sample file.** `examples/data/real-data/abaqus/1.inp` is a **94 MB** deck describing a large **3-D solid**
part. (It turns out to be a **vertebra** — a spine bone — as the plots will show.)

## 1. Read the file

## Getting the data

The sample file(s) this notebook reads are **not stored in the repository** (they are large and carry per-source licensing). The cell below downloads just this driver's data from the project's public **Google Drive** folder ([link](https://drive.google.com/drive/folders/16IkuKUkCo21uDb_X1iZciSPp7X99bmjv)) into `examples/data/real-data/abaqus/` — but only if it is not already there, so it is a no-op once you have the data. It uses the standard library only.

In [ ]:
# --- Getting the data (download it if you do not already have it) ------------
# The real-data samples are not committed to the repository (large, per-source
# licensing). If the file this notebook needs is not already under
# examples/data/real-data/, download it from the project's public Google Drive
# folder below. Standard library only.
#   https://drive.google.com/drive/folders/16IkuKUkCo21uDb_X1iZciSPp7X99bmjv
from pathlib import Path
import urllib.request, urllib.parse, http.cookiejar, re, html, zipfile

DATA = Path("../../../examples/data/real-data")

def _download_drive(file_id, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    op = urllib.request.build_opener(urllib.request.HTTPCookieProcessor(http.cookiejar.CookieJar()))
    op.addheaders = [("User-Agent", "Mozilla/5.0")]
    base = "https://drive.usercontent.google.com/download"
    r = op.open(base + "?" + urllib.parse.urlencode({"id": file_id, "export": "download", "confirm": "t"}))
    if r.headers.get_content_type() == "text/html":          # virus-scan page -> re-submit its form
        form = dict(re.findall(r'name="([^"]+)"\s+value="([^"]*)"', r.read().decode("utf-8", "replace")))
        r = op.open(base + "?" + urllib.parse.urlencode(form))
    with open(dest, "wb") as fh:
        while True:
            chunk = r.read(1 << 20)
            if not chunk:
                break
            fh.write(chunk)

marker = DATA / "abaqus" / "1.inp"
if not marker.exists():
    print("Downloading abaqus sample (~99 MB) from Google Drive ...")
    _download_drive("1GwU2tuLV6JYiJz8MAG0mDbgnag1KKWxL", marker)
    print("Saved:", marker)
else:
    print("abaqus data already present:", marker)


In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from cleopatra.config import Config

from pyramids.netcdf.ugrid import MeshDataset

DATA = Path("../../../examples/data/real-data")
inp = DATA / "abaqus" / "1.inp"

t0 = time.perf_counter()
ds = MeshDataset.read_file(inp, driver="abaqus")
secs = time.perf_counter() - t0
Config.set_matplotlib_backend()   # cleopatra picks the inline backend for the plots below

print(f"read in {secs:.1f} s")
type(ds).__name__

## 2. Explore the mesh — attributes

A `MeshDataset` describes the mesh through simple attributes: size (`n_node`, `n_face`), coordinate system
(`crs` / `epsg`), extent (`bounds`), and attached data fields (`variable_names`).

In [ ]:
pd.Series({
    "n_node": ds.n_node,
    "n_face": ds.n_face,
    "crs": ds.crs,
    "epsg": ds.epsg,
    "bounds": tuple(round(b, 1) for b in ds.bounds),
    "variable_names": ds.variable_names,
})

**752 666 nodes**, **332 200 faces**. `crs`/`epsg` are `None` — an engineering model has no map
projection, just model length units — so there is no basemap or reprojection. There is **one data field,
`node_z`**.

## 3. How faces are built — the connectivity

`ds.mesh` is the underlying `Mesh2d`; its `face_node_connectivity` says which nodes make each face.

In [ ]:
conn = ds.mesh.face_node_connectivity.data
print("connectivity shape:", conn.shape)       # (n_face, k)
print("nodes per face (k):", conn.shape[1])

`k = 8` means each element has 8 nodes — **hexahedra (bricks)**, the building block of a 3-D *solid* mesh
(a flat 2-D mesh would be `k = 3` triangles or `k = 4` quads).

## 4. What is `node_z`? — the data we will plot

**`node_z` is the height of each node** — its position along the vertical (third) axis. The part is a 3-D
object, but a `MeshDataset` lays the mesh out in the flat **x–y plane**, so the third dimension is carried
separately as this per-node field. Every plot below **colours the mesh by `node_z`**, so *colour = height*,
which lets a flat picture show the 3-D shape.

`ds.summary()` gives per-field statistics; `ds.masked_values(name)` returns the field as a NumPy array.

In [ ]:
print(ds.summary())
z = ds.masked_values("node_z")
print("\nnode_z (height): min %.0f, max %.0f  (model length units)" % (z.min(), z.max()))

The height ranges from about **1099 to 1666** — a real 3-D relief, not a flat plate.

## 5. Plot the whole part — `ds.plot("node_z")`

Plotting goes through pyramids: `ds.plot(variable)` renders the mesh with cleopatra and returns a **`MeshGlyph`**
(its `.ax` / `.fig` are the matplotlib axes/figure). We view the part from above (x–y) and **colour each cell by
its height `node_z`** — so the colour scale on the right is height in model units.

In [ ]:
glyph = ds.plot("node_z", cmap="terrain", colorbar=True,
                title="The whole part, coloured by height (node_z)")
glyph.ax.set_aspect("equal")

**What you are looking at:** the part from directly overhead, with colour = height. The shape is a
**vertebra**: the pale, **high** dome on the right is the vertebral **body**; the darker, **lower** arms
reaching to the left are the **transverse processes**; the gap between them is the vertebral foramen (the hole
the spinal cord passes through). Reading the colours tells you how the bone rises and falls in 3-D.

## 6. Zoom in with `crop()`

`crop(bbox=...)` returns a **new** `MeshDataset` containing only the faces inside a box — here a box around one
of the transverse processes. It behaves exactly like the full dataset, so we plot it the same way.

In [ ]:
process = ds.crop(bbox=(-80, 20, -30, 60))
print("faces in the whole part:", ds.n_face, " -> in this piece:", process.n_face)

glyph = process.plot("node_z", cmap="terrain", colorbar=True,
                     title="One transverse process — coloured by height")
glyph.ax.set_aspect("equal")

**What you are looking at:** a close-up of **one transverse process** (a bony arm of the vertebra), still
coloured by height `node_z`. The surface is **highest (pale)** where the arm joins the body on the right and
**drops (blue) toward the tip** on the left — the colour gradient traces the real 3-D slope of that arm. This
is the same height data as the full plot, just cropped to one feature.

## Takeaway

- The `abaqus` driver reads a large FEA `.inp` deck (**94 MB, 752 666 nodes**) into a pyramids `MeshDataset` in
  a few seconds.
- The one data field, **`node_z`**, is the **height** of each point; every plot colours the mesh by it, so a
  flat image shows the 3-D shape (here, a vertebra).
- Explore with attributes (`n_node`, `bounds`, `variable_names`) and methods (`summary()`,
  `masked_values()`, `face_node_connectivity`, `crop()`), and plot with `ds.plot(...)` (a cleopatra `MeshGlyph`).